# Markov-Chain and Dwell-Time Model for SWaT
This notebook implements a state-based anomaly detection pipeline using a Markov-Chain for sequence likelihood and a Dwell-Time model for state duration violations. It includes both 1st-order and 2nd-order Markov chains for comparison.

In [5]:
import pandas as pd
import numpy as np
import os
import joblib
import time
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Parameters
STAGE_NUM = 1
NORMAL_DATA_PATH = f'dataset/normal_stage{STAGE_NUM}.csv'
ATTACK_DATA_PATH = f'dataset/attack_stage{STAGE_NUM}.csv'
MODEL_SAVE_PATH = f'models/markov_stage{STAGE_NUM}.pkl'

K_BINS = 5
DWELL_TIME_PERCENTILE = 99.0
MIN_SAMPLES_DWELL = 10
WINDOW_SIZE = 30


## 1. Preprocessing
- Parses timestamps, sorts by time, and coercs columns to numeric.
- Forward-fills NaNs to avoid look-ahead.
- Identifies discrete vs continuous columns.
- Discretizes continuous sensors into `K_BINS` equal-frequency bins using normal data only.
- Concatenates state representations into a single string label.

In [6]:
def load_and_preprocess(filepath, is_attack=False):
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.strip()
    
    df['Timestamp'] = pd.to_datetime(df['Timestamp'].str.strip(), format='%d/%m/%Y %I:%M:%S %p', errors='coerce')
    df = df.sort_values('Timestamp').reset_index(drop=True)
    
    skip_cols = ['Timestamp']
    if 'attack' in df.columns:
        skip_cols.append('attack')
        
    for col in df.columns:
        if col not in skip_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            
    df = df.ffill().fillna(0)
    
    if not is_attack and 'attack' in df.columns:
        df['attack'] = 0
        
    return df

def identify_column_types(df, skip_cols):
    features = [c for c in df.columns if c not in skip_cols]
    discrete_cols, continuous_cols = [], []
    for col in features:
        if df[col].nunique() <= 3:
            discrete_cols.append(col)
        else:
            continuous_cols.append(col)
    return discrete_cols, continuous_cols

def discretize_and_combine(df_normal, df_attack, continuous_cols, discrete_cols, k_bins):
    df_normal = df_normal.copy()
    df_attack = df_attack.copy()
    bin_edges_dict = {}
    
    if continuous_cols:
        est = KBinsDiscretizer(n_bins=k_bins, encode='ordinal', strategy='quantile', subsample=None)
        df_normal[continuous_cols] = est.fit_transform(df_normal[continuous_cols])
        df_attack[continuous_cols] = est.transform(df_attack[continuous_cols])
        for i, col in enumerate(continuous_cols):
            bin_edges_dict[col] = est.bin_edges_[i]
            
    features = discrete_cols + continuous_cols
    
    # Vectorized string join
    df_normal['System_State'] = df_normal[features[0]].astype(int).astype(str)
    df_attack['System_State'] = df_attack[features[0]].astype(int).astype(str)
    
    for col in features[1:]:
        df_normal['System_State'] += '|' + df_normal[col].astype(int).astype(str)
        df_attack['System_State'] += '|' + df_attack[col].astype(int).astype(str)
        
    return df_normal, df_attack, bin_edges_dict

df_normal = load_and_preprocess(NORMAL_DATA_PATH, is_attack=False)
df_attack = load_and_preprocess(ATTACK_DATA_PATH, is_attack=True)

skip_cols = ['Timestamp', 'attack']
discrete_cols, continuous_cols = identify_column_types(df_normal, skip_cols)
df_normal, df_attack, bin_edges = discretize_and_combine(df_normal, df_attack, continuous_cols, discrete_cols, K_BINS)

print(f"Normal data shape: {df_normal.shape}, Unique states: {df_normal['System_State'].nunique()}")
print(f"Attack data shape: {df_attack.shape}")


Normal data shape: (495000, 8), Unique states: 57
Attack data shape: (449919, 8)


## 2. Markov Chain Training
Builds transition probability matrices (1st and 2nd order) with Laplace smoothing (`alpha=1`) applied on normal data.

In [7]:
def train_markov_chain(states, alpha=1, order=1):
    unique_states = list(set(states))
    s2i = {s: i for i, s in enumerate(unique_states)}
    n_states = len(unique_states)
    
    if order == 1:
        counts = np.ones((n_states, n_states)) * alpha
        for i in range(len(states) - 1):
            counts[s2i[states[i]], s2i[states[i+1]]] += 1
        log_p = np.log(counts / counts.sum(axis=1, keepdims=True))
        return s2i, log_p, unique_states
        
    elif order == 2:
        counts = {}
        for i in range(len(states) - 2):
            h = (states[i], states[i+1])
            nxt = states[i+2]
            if h not in counts:
                counts[h] = {}
            counts[h][nxt] = counts[h].get(nxt, 0) + 1
            
        log_p = {}
        for h, nxt_counts in counts.items():
            total = sum(nxt_counts.values()) + n_states * alpha
            log_p[h] = {nxt: np.log((c + alpha) / total) for nxt, c in nxt_counts.items()}
            log_p[h]['__unseen__'] = np.log(alpha / total)
            
        fallback_prob = np.log(1.0 / n_states)
        return s2i, (log_p, fallback_prob), unique_states

normal_states = df_normal['System_State'].values
s2i_1, log_p_1, unq_1 = train_markov_chain(normal_states, order=1)
s2i_2, log_p_2, unq_2 = train_markov_chain(normal_states, order=2)
print("Markov Chains (1st and 2nd order) trained.")


Markov Chains (1st and 2nd order) trained.


## 3. Dwell-Time Distribution
Computes empirical dwell-time distributions per state on normal data. Uses the 99th percentile as the bound, with a global fallback for rare states.

In [8]:
def compute_dwell_times(states):
    dwell_durations = {}
    all_durations = []
    
    if len(states) == 0: return {}, []
    
    curr = states[0]
    duration = 1
    for s in states[1:]:
        if s == curr:
            duration += 1
        else:
            dwell_durations.setdefault(curr, []).append(duration)
            all_durations.append(duration)
            curr = s
            duration = 1
            
    dwell_durations.setdefault(curr, []).append(duration)
    all_durations.append(duration)
    return dwell_durations, all_durations

def build_dwell_time_bounds(dwell_durations, all_durations, percentile, min_samples):
    global_bound = np.percentile(all_durations, percentile) if all_durations else 1
    dwell_bounds = {}
    
    for s, durs in dwell_durations.items():
        if len(durs) >= min_samples:
            dwell_bounds[s] = {'mean': np.mean(durs), 'std': np.std(durs), 'bound': np.percentile(durs, percentile)}
        else:
            dwell_bounds[s] = {'mean': np.mean(durs), 'std': np.std(durs), 'bound': global_bound}
    return dwell_bounds, global_bound

dwell_durations, all_durations = compute_dwell_times(normal_states)
dwell_bounds, global_bound = build_dwell_time_bounds(dwell_durations, all_durations, DWELL_TIME_PERCENTILE, MIN_SAMPLES_DWELL)
print(f"Global dwell time {DWELL_TIME_PERCENTILE}th percentile bound: {global_bound}")


Global dwell time 99.0th percentile bound: 10.0


## 4. Sequence Likelihood Scoring & Normalization
Scores windows of length `WINDOW_SIZE` by accumulating log transition probabilities. Negative Log-Likelihood (NLL) is Min-Max scaled based on normal data ranges. The anomaly threshold is defined as the 99th percentile of normal NLLs.

In [9]:
def score_sequence_window(states, s2i, model_params, window_size, order=1, alpha=1):
    nll_scores = np.zeros(len(states))
    step_lls = np.zeros(len(states))
    n_states = len(s2i)
    
    if order == 1:
        log_p = model_params
        unseen_prob = np.log(alpha / (alpha * n_states))
        for i in range(1, len(states)):
            prev, curr = states[i-1], states[i]
            if prev in s2i and curr in s2i:
                step_lls[i] = log_p[s2i[prev], s2i[curr]]
            else:
                step_lls[i] = unseen_prob
    elif order == 2:
        log_p, fallback_prob = model_params
        for i in range(2, len(states)):
            hist = (states[i-2], states[i-1])
            curr = states[i]
            if hist in log_p:
                step_lls[i] = log_p[hist].get(curr, log_p[hist]['__unseen__'])
            else:
                step_lls[i] = fallback_prob
                
    window_sums = pd.Series(step_lls).rolling(window=window_size, min_periods=1).sum().values
    return -window_sums, step_lls

def normalize_scores(scores, min_val, max_val):
    return np.clip((scores - min_val) / (max_val - min_val + 1e-9), 0, None)

# Score on normal data (Order 1)
normal_nll_1, _ = score_sequence_window(normal_states, s2i_1, log_p_1, WINDOW_SIZE, order=1)
nll_min_1, nll_max_1 = normal_nll_1.min(), normal_nll_1.max()
norm_nll_1 = normalize_scores(normal_nll_1, nll_min_1, nll_max_1)
THRESH_1 = np.percentile(norm_nll_1, 99)

# Score on normal data (Order 2)
normal_nll_2, _ = score_sequence_window(normal_states, s2i_2, log_p_2, WINDOW_SIZE, order=2)
nll_min_2, nll_max_2 = normal_nll_2.min(), normal_nll_2.max()
norm_nll_2 = normalize_scores(normal_nll_2, nll_min_2, nll_max_2)
THRESH_2 = np.percentile(norm_nll_2, 99)

print(f"Order 1 - Normalization Bounds: [{nll_min_1:.2f}, {nll_max_1:.2f}], Threshold: {THRESH_1:.3f}")
print(f"Order 2 - Normalization Bounds: [{nll_min_2:.2f}, {nll_max_2:.2f}], Threshold: {THRESH_2:.3f}")


Order 1 - Normalization Bounds: [-0.00, 81.65], Threshold: 0.482
Order 2 - Normalization Bounds: [-0.00, 73.80], Threshold: 0.526


## 5. Anomaly Decision Fusion & Inference timing
A timestep is flagged if NLL exceeds threshold OR dwell time exceeds its bound. We also explicitly measure the inference latency row-by-row to reflect edge-hardware usage.

In [10]:
def compute_dwell_violations(states, dwell_bounds, global_bound):
    violations = np.zeros(len(states), dtype=bool)
    current_duration = 1
    for i in range(1, len(states)):
        if states[i] == states[i-1]:
            current_duration += 1
        else:
            current_duration = 1
        bound = dwell_bounds.get(states[i], {}).get('bound', global_bound)
        if current_duration > bound:
            violations[i] = True
    return violations

def final_anomaly_decision(nll_scores, dwell_violations, likelihood_threshold):
    '''Explicit combination logic: OR operation'''
    likelihood_anomalies = nll_scores > likelihood_threshold
    return likelihood_anomalies | dwell_violations

def measure_inference_latency(states, s2i, log_p, window_size, d_bounds, g_bound, thresh, n_min, n_max, n_samples=1000, order=1):
    latencies = []
    n_samples = min(n_samples, len(states) - window_size)
    n_states = len(s2i)
    unseen_prob = np.log(1 / (1 * n_states))
    
    for i in range(window_size, window_size + n_samples):
        start_time = time.time()
        curr_state = states[i]
        
        # Dwell time
        duration = 1
        for j in range(i-1, -1, -1):
            if states[j] == curr_state: duration += 1
            else: break
        bound = d_bounds.get(curr_state, {}).get('bound', g_bound)
        dwell_anomaly = duration > bound
        
        # Sequence likelihood
        window = states[i-window_size+1 : i+1]
        nll = 0
        if order == 1:
            for j in range(1, len(window)):
                s_prev, s_curr = window[j-1], window[j]
                if s_prev in s2i and s_curr in s2i:
                    nll -= log_p[s2i[s_prev], s2i[s_curr]]
                else: nll -= unseen_prob
        
        nll_norm = max(0, (nll - n_min) / (n_max - n_min + 1e-9))
        seq_anomaly = nll_norm > thresh
        
        final_decision = dwell_anomaly or seq_anomaly
        latencies.append((time.time() - start_time) * 1000) # ms
        
    return np.mean(latencies), np.percentile(latencies, 95)

attack_states = df_attack['System_State'].values

# Order 1 Inference
attack_nll_1, _ = score_sequence_window(attack_states, s2i_1, log_p_1, WINDOW_SIZE, order=1)
attack_nll_norm_1 = normalize_scores(attack_nll_1, nll_min_1, nll_max_1)
attack_dwell_violations = compute_dwell_violations(attack_states, dwell_bounds, global_bound)
y_pred_1 = final_anomaly_decision(attack_nll_norm_1, attack_dwell_violations, THRESH_1)

mean_lat_1, p95_lat_1 = measure_inference_latency(attack_states, s2i_1, log_p_1, WINDOW_SIZE, dwell_bounds, global_bound, THRESH_1, nll_min_1, nll_max_1, order=1)

# Order 2 Inference
attack_nll_2, _ = score_sequence_window(attack_states, s2i_2, log_p_2, WINDOW_SIZE, order=2)
attack_nll_norm_2 = normalize_scores(attack_nll_2, nll_min_2, nll_max_2)
y_pred_2 = final_anomaly_decision(attack_nll_norm_2, attack_dwell_violations, THRESH_2)


## 6. Evaluation
Evaluate Order 1 vs Order 2 models, showing confusion matrices, classification reports, and latency measurements.

In [11]:
def evaluate_predictions(y_true, y_pred, y_score, stage_num, order_name, mean_lat=None, p95_lat=None):
    print(f"\n--- Stage {stage_num} Evaluation ({order_name}) ---")
    if len(np.unique(y_true)) > 1:
        cm = confusion_matrix(y_true, y_pred)
        print("Confusion Matrix:")
        print(cm)
        print("\nClassification Report:")
        print(classification_report(y_true, y_pred, digits=4))
        
        auc = roc_auc_score(y_true, y_score)
        print(f"ROC-AUC Score: {auc:.4f}")
        
        tn, fp, fn, tp = cm.ravel()
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
        print(f"False Positive Rate: {fpr:.4f}")
        print(f"False Negative Rate: {fnr:.4f}")
        
    else:
        print(f"Stage {stage_num} has no attacks (or only attacks). Skipping supervised metrics.")
        print("Summary stats for normalized NLL anomaly score:")
        print(f"Mean: {y_score.mean():.4f}, Std: {y_score.std():.4f}, Max: {y_score.max():.4f}")
        
        plt.figure(figsize=(15, 3))
        plt.plot(y_score, label=f"Anomaly Score ({order_name})")
        plt.axhline(THRESH_1 if "1st" in order_name else THRESH_2, color='r', linestyle='--', label="Threshold")
        plt.title(f"Stage {stage_num} Anomaly Scores over time")
        plt.legend()
        plt.show()
        
    if mean_lat is not None and p95_lat is not None:
        print(f"\nInference Latency ({order_name}): Mean = {mean_lat:.4f} ms, p95 = {p95_lat:.4f} ms")

if 'attack' in df_attack.columns:
    y_true = df_attack['attack'].values
    evaluate_predictions(y_true, y_pred_1, attack_nll_norm_1, STAGE_NUM, "1st Order", mean_lat_1, p95_lat_1)
    evaluate_predictions(y_true, y_pred_2, attack_nll_norm_2, STAGE_NUM, "2nd Order")
else:
    print("No ground truth 'attack' column found for evaluation.")



--- Stage 1 Evaluation (1st Order) ---
Confusion Matrix:
[[388083  55167]
 [  3297   3372]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9916    0.8755    0.9300    443250
           1     0.0576    0.5056    0.1034      6669

    accuracy                         0.8701    449919
   macro avg     0.5246    0.6906    0.5167    449919
weighted avg     0.9777    0.8701    0.9177    449919

ROC-AUC Score: 0.8338
False Positive Rate: 0.1245
False Negative Rate: 0.4944

Inference Latency (1st Order): Mean = 0.3428 ms, p95 = 0.8057 ms

--- Stage 1 Evaluation (2nd Order) ---
Confusion Matrix:
[[386759  56491]
 [  3246   3423]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9917    0.8726    0.9283    443250
           1     0.0571    0.5133    0.1028      6669

    accuracy                         0.8672    449919
   macro avg     0.5244    0.6929    0.5156    449919
weighted avg     0.9778

## 7. Artifact Persistence
Save configuration and transition matrices for production.

In [12]:
os.makedirs('models', exist_ok=True)
model_artifacts = {
    's2i_1': s2i_1, 'log_p_1': log_p_1, 'unq_1': unq_1,
    's2i_2': s2i_2, 'log_p_2': log_p_2, 'unq_2': unq_2,
    'dwell_bounds': dwell_bounds, 'global_bound': global_bound,
    'bin_edges': bin_edges,
    'nll_min_1': nll_min_1, 'nll_max_1': nll_max_1, 'thresh_1': THRESH_1,
    'nll_min_2': nll_min_2, 'nll_max_2': nll_max_2, 'thresh_2': THRESH_2,
    'discrete_cols': discrete_cols, 'continuous_cols': continuous_cols,
    'k_bins': K_BINS, 'window_size': WINDOW_SIZE
}
joblib.dump(model_artifacts, MODEL_SAVE_PATH)
print(f"Saved model artifacts to {MODEL_SAVE_PATH}")


Saved model artifacts to models/markov_stage1.pkl


## 8. Full Pipeline Run (Stages 1-6)
Evaluates both 1st-order and 2nd-order pipelines across all stages.

In [ ]:
results = []

for stage in range(1, 7):
    norm_path = f'dataset/normal_stage{stage}.csv'
    attk_path = f'dataset/attack_stage{stage}.csv'
    
    if not os.path.exists(norm_path) or not os.path.exists(attk_path):
        continue
        
    df_n = load_and_preprocess(norm_path, is_attack=False)
    df_a = load_and_preprocess(attk_path, is_attack=True)
    
    disc_c, cont_c = identify_column_types(df_n, skip_cols)
    df_n, df_a, _ = discretize_and_combine(df_n, df_a, cont_c, disc_c, K_BINS)
    
    # Train 1st Order
    s2i_1, log_p_1, _ = train_markov_chain(df_n['System_State'].values, order=1)
    norm_nll_1, _ = score_sequence_window(df_n['System_State'].values, s2i_1, log_p_1, WINDOW_SIZE, order=1)
    n_min_1, n_max_1 = norm_nll_1.min(), norm_nll_1.max()
    t_1 = np.percentile(normalize_scores(norm_nll_1, n_min_1, n_max_1), 99)
    
    # Train 2nd Order
    s2i_2, log_p_2, _ = train_markov_chain(df_n['System_State'].values, order=2)
    norm_nll_2, _ = score_sequence_window(df_n['System_State'].values, s2i_2, log_p_2, WINDOW_SIZE, order=2)
    n_min_2, n_max_2 = norm_nll_2.min(), norm_nll_2.max()
    t_2 = np.percentile(normalize_scores(norm_nll_2, n_min_2, n_max_2), 99)
    
    # Dwell Time
    dwell_d, all_d = compute_dwell_times(df_n['System_State'].values)
    d_bounds, g_bound = build_dwell_time_bounds(dwell_d, all_d, DWELL_TIME_PERCENTILE, MIN_SAMPLES_DWELL)
    
    # Infer
    attk_states = df_a['System_State'].values
    y_t = df_a['attack'].values if 'attack' in df_a.columns else np.zeros(len(attk_states))
    has_attacks = len(np.unique(y_t)) > 1
    
    a_dwell = compute_dwell_violations(attk_states, d_bounds, g_bound)
    
    for order_name, s2i, log_p, order_idx, n_min, n_max, t in [("1st Order", s2i_1, log_p_1, 1, n_min_1, n_max_1, t_1), 
                                                               ("2nd Order", s2i_2, log_p_2, 2, n_min_2, n_max_2, t_2)]:
        a_nll, _ = score_sequence_window(attk_states, s2i, log_p, WINDOW_SIZE, order=order_idx)
        a_nll_norm = normalize_scores(a_nll, n_min, n_max)
        pred = final_anomaly_decision(a_nll_norm, a_dwell, t)
        
        # Calculate Latency for 1st order (as representative)
        lat_mean, lat_p95 = np.nan, np.nan
        if order_idx == 1:
            lat_mean, lat_p95 = measure_inference_latency(attk_states, s2i, log_p, WINDOW_SIZE, d_bounds, g_bound, t, n_min, n_max, order=1)
        
        if has_attacks:
            cm = confusion_matrix(y_t, pred)
            tn, fp, fn, tp = cm.ravel()
            rep = classification_report(y_t, pred, output_dict=True, zero_division=0)
            auc = roc_auc_score(y_t, a_nll_norm)
            
            results.append({
                'Stage': stage, 'Model': order_name, 'Precision': rep['1']['precision'], 
                'Recall': rep['1']['recall'], 'F1': rep['1']['f1-score'], 'ROC-AUC': auc,
                'FPR': fp / (fp + tn) if (fp+tn)>0 else 0, 'FNR': fn / (fn + tp) if (fn+tp)>0 else 0,
                'Lat Mean (ms)': lat_mean, 'Lat p95 (ms)': lat_p95
            })
        else:
            results.append({
                'Stage': stage, 'Model': order_name, 'Precision': np.nan, 'Recall': np.nan, 
                'F1': np.nan, 'ROC-AUC': np.nan, 'FPR': np.nan, 'FNR': np.nan,
                'Lat Mean (ms)': lat_mean, 'Lat p95 (ms)': lat_p95
            })

df_results = pd.DataFrame(results)
print("\n=== Cross-Stage Summary ===")
display(df_results)
